# 长期记忆Agent
## 在工具中访问长期记忆
1. key放在agent的invoke的消息中
2. create_agent(store=InMemoryStore())设置store
3. 重写state_schema类，继承AgentState
4. 工具增加runtime: ToolRuntime参数，用于获取消息中的key

### 基于内存的工具长期记忆

In [1]:

from langchain_core.tools import tool
from typing import NotRequired
from langgraph.prebuilt import ToolRuntime
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState

from common import init_simple_dashscope_model, load_postgresql_url

store = InMemoryStore()

# 自定义一个继承于AgentState的类
class CustomState(AgentState):
    user_id : NotRequired[str]

# 保存用户信息到长期记忆中
@tool(parse_docstring=True)
def save_user_info(name : str , runtime : ToolRuntime) -> str:
    """
    将客户信息保存在长期记忆中

    Args:
        name : 用户名
        runtime : 工具的运行时

    Returns:
        str : 保存状态
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name" : name}

    runtime.store.put(namespace, key, value)
    return "saved"


@tool(parse_docstring=True)
def get_user_info(runtime : ToolRuntime) -> str:
    """
    从长期记忆中读取客户的信息

    Args:
        runtime : 工具的运行时

    Returns:
        str : 用户信息
    """
    namespace = ("users",)
    key = runtime.state["user_id"]

    item = runtime.store.get(namespace,key)
    return str(item.value) if item else "unknown"

agent = create_agent(
    model=init_simple_dashscope_model('qwen-max'),
    tools=[save_user_info,get_user_info],
    store=store,
    state_schema=CustomState,
    system_prompt="用户提及个人信息时，可以使用工具保存用户信息。如果用户询问个人信息时，可以尝试使用工具读取用户信息"
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()



============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_561ca691150f498f8ea4fe)
 Call ID: call_561ca691150f498f8ea4fe
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

很高兴认识你，小花！我已经记住了你的名字。有什么可以帮助你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_2163d5aeac6f4f0986bf81)
 Call ID: call_2163d5aeac6f4f0986bf81
  Args:
================================= Tool Message ======

## 基于Postgres的工具长期记忆

In [2]:

from langgraph.store.postgres import PostgresStore
from langgraph.prebuilt import ToolRuntime
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState



# 自定义一个继承于AgentState的类
class CustomState(AgentState):
    user_id : NotRequired[str]

# 保存用户信息到长期记忆中
@tool(parse_docstring=True)
def save_user_info(name : str , runtime : ToolRuntime) -> str:
    """
    将客户信息保存在长期记忆中

    Args:
        name : 用户名
        runtime : 工具的运行时

    Returns:
        str : 保存状态
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name" : name}

    runtime.store.put(namespace, key, value)
    return "saved"


@tool(parse_docstring=True)
def get_user_info(runtime : ToolRuntime) -> str:
    """
    从长期记忆中读取客户的信息

    Args:
        runtime : 工具的运行时

    Returns:
        str : 用户信息
    """
    namespace = ("users",)
    key = runtime.state["user_id"]

    item = runtime.store.get(namespace,key)
    return str(item.value) if item else "unknown"


with PostgresStore.from_conn_string(load_postgresql_url()) as store:
    store.setup()

    agent = create_agent(
        model=init_simple_dashscope_model('qwen-max'),
        tools=[save_user_info,get_user_info],
        store=store,
        state_schema=CustomState,
        system_prompt="用户提及个人信息时，可以使用工具保存用户信息。如果用户询问个人信息时，可以尝试使用工具读取用户信息"
    )

    print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
    response1 = agent.invoke({
        "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
        "user_id": "user-1"
    })
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
    response2 = agent.invoke({
        "messages": [HumanMessage("我是谁")],
        "user_id": "user-1"
    })
    for msg in response2["messages"]:
        msg.pretty_print()


NameError: name 'load_postgresql_url' is not defined